# Exploración del log del bot

> **Este notebook es solo para exploración.** La lógica de producción vive en `src/`.
> Nada de lo que se escriba aquí debe formar parte del pipeline.

Aquí se documentan los hallazgos sobre el formato del log que condicionaron el diseño
del pipeline, y la validación de las reglas de negocio contra los datos reales.

Requiere que haya archivos `.log` en `data/input/`.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

# Permite importar `src` estando dentro de notebooks/
sys.path.insert(0, str(Path.cwd().parent))

from src.config import ACCIONES, DIR_ENTRADA
from src.extract.log_reader import iter_operaciones, iter_registros
from src.transform.parser import parsear
from src.transform.rules import evaluar
from src.utils.texto import empieza_con_alguno, normalizar

accion = ACCIONES["reseteo_usuario"]
logs = sorted(DIR_ENTRADA.glob("*.log"))
print(f"{len(logs)} archivos de log:")
for ruta in logs:
    print(f"  {ruta.name}: {ruta.stat().st_size / 1024:,.0f} KB")

## Hallazgo 1: un registro puede ocupar varias líneas

El volcado `Raw Response:` de ADManager continúa en líneas que **no repiten el
`operation_Id`**. Un `grep` por identificador pierde justamente los datos del usuario
(nombre, oficina, OU, descripción).

In [ ]:
for registro in iter_registros(logs[0]):
    if "Raw Response:" in registro and registro.count("\n") > 1:
        print(f"El registro ocupa {registro.count(chr(10))} líneas:\n")
        for i, linea in enumerate(registro.splitlines()):
            marca = "[con operation_Id]" if "operation_Id" in linea else "[SIN operation_Id]"
            print(f"  línea {i} {marca}: {linea[:110]}")
        break

## Hallazgo 2: las operaciones se intercalan

El bot atiende peticiones concurrentes, así que los registros de una misma operación
**no son contiguos**. Agrupar por cercanía fragmentaría casi una de cada cuatro.

In [ ]:
import re

PATRON = re.compile(r"operation_Id=([0-9a-f]+)")

for ruta in logs:
    bloques = []
    for registro in iter_registros(ruta):
        encontrado = PATRON.search(registro)
        if encontrado and (not bloques or bloques[-1] != encontrado.group(1)):
            bloques.append(encontrado.group(1))
    partidas = sum(1 for _, n in Counter(bloques).items() if n > 1)
    print(
        f"{ruta.name}: {len(set(bloques)):4d} operaciones, {partidas:4d} partidas en varios bloques"
    )

## Parseo completo y distribución de resultados

In [ ]:
operaciones = []
for ruta in logs:
    for cruda in iter_operaciones(ruta, accion):
        operacion = parsear(cruda, accion)
        if operacion:
            operaciones.append(operacion)

print(f"Total de operaciones de reseteo: {len(operaciones)}")
print(f"operation_Id únicos            : {len({o.id for o in operaciones})}")
print()
for status, n in Counter(o.status_code for o in operaciones).most_common():
    print(f"  HTTP {status}: {n:5d}")

## Validación de las reglas del 403

Es la comprobación más importante del proyecto, y tiene dos mitades:

1. **¿Explican las reglas todos los rechazos?** Cada 403 debe encajar en alguna causa.
2. **¿Producen falsos positivos?** Ninguna operación exitosa debería disparar una regla.

Si alguna de las dos falla, las reglas no reproducen el comportamiento del bot.

In [ ]:
from src.config import OU_RESTRINGIDA, PREFIJOS_AUTORIZADOS


def causas(operacion):
    """Razones por las que el bot debería denegar esta operación."""
    solicitante, target = operacion.usuario_solicitante, operacion.usuario_target
    if solicitante is None or target is None:
        return ["(sin datos de usuario)"]

    encontradas = []
    if not empieza_con_alguno(solicitante.descripcion, PREFIJOS_AUTORIZADOS):
        encontradas.append("no es gerente ni admin")
    if normalizar(solicitante.oficina) != normalizar(target.oficina):
        encontradas.append("distinta oficina")
    if normalizar(target.ou_name) == OU_RESTRINGIDA:
        encontradas.append("OU restringida")
    return encontradas


rechazos = [o for o in operaciones if o.status_code == 403]
exitos = [o for o in operaciones if o.status_code == 200]

print(f"1) Rechazos (403): {len(rechazos)}")
for motivo, n in Counter(" + ".join(causas(o)) or "SIN CAUSA (!)" for o in rechazos).most_common():
    print(f"     {n:4d}  {motivo}")

falsos_positivos = [o for o in exitos if causas(o)]
print(f"\n2) Éxitos (200) que dispararían una regla: {len(falsos_positivos)} de {len(exitos)}")

## Dominio de los campos usados por las reglas

`OFFICE` no siempre es numérico y no puede tratarse como entero: se perdería el cero
a la izquierda de códigos como `0520`.

In [ ]:
usuarios = [u for o in operaciones for u in (o.usuario_solicitante, o.usuario_target) if u]

oficinas = Counter(u.oficina for u in usuarios)
no_numericas = {k: v for k, v in oficinas.items() if k and not k.isdigit()}
print(f"OFFICE distintos: {len(oficinas)}")
print(f"OFFICE no numéricos: {no_numericas}")
print(f"OFFICE con 'corporativo': {[o for o in oficinas if 'corporativo' in normalizar(o)]}")

print("\nOU_NAME distintos:")
for ou, n in Counter(u.ou_name for u in usuarios).most_common():
    marca = "  <-- OU RESTRINGIDA" if normalizar(ou) == OU_RESTRINGIDA else ""
    print(f"  {n:5d}  {ou}{marca}")

print("\nDESCRIPTION que autorizan (empiezan por gerente/admin):")
autorizadas = Counter(
    u.descripcion for u in usuarios if empieza_con_alguno(u.descripcion, PREFIJOS_AUTORIZADOS)
)
for desc, n in autorizadas.most_common(10):
    print(f"  {n:5d}  {desc}")

## Timeouts (504)

La especificación fija el umbral en 35 segundos. Se comprueba contra la duración real
de cada operación.

Ojo: **la clasificación la da el código de respuesta, no la duración.** Hay operaciones
exitosas que también superan los 35 s.

In [ ]:
for status in sorted({o.status_code for o in operaciones}):
    duraciones = sorted(o.duracion_segundos for o in operaciones if o.status_code == status)
    lentas = sum(1 for d in duraciones if d > 35)
    mediana = duraciones[len(duraciones) // 2]
    print(
        f"  HTTP {status}: n={len(duraciones):5d}  mediana={mediana:6.2f}s  "
        f"max={duraciones[-1]:6.2f}s  >35s={lentas}"
    )

## Resultados finales generados

In [ ]:
for resultado, n in Counter(evaluar(o) for o in operaciones).most_common():
    print(f"{n:5d}  {resultado[:120]}")

## Conclusiones que se llevaron al diseño

| Hallazgo | Cómo lo resuelve el pipeline |
|---|---|
| Los registros ocupan varias líneas | `iter_registros` abre registro nuevo solo ante una marca de tiempo |
| Las operaciones se intercalan | `iter_operaciones` acumula por `operation_Id`, no por cercanía |
| El `operation_Id` es único de forma global | Se usa como columna `id` y base de la idempotencia |
| El orden de las búsquedas varía | Se cruza por el `filter`, nunca por posición |
| Los usuarios llegan URL-encoded | `parse_qs` los decodifica antes de cruzarlos |
| `OFFICE` es texto, a veces no numérico | Se compara normalizado y nunca se convierte a entero |
| No hay ejemplos de 202 ni 429 | Sus reglas se cubren con pruebas sintéticas |